<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Misc/Turn_of_Month.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import datetime
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

In [18]:
# ==========================================
# PARAMETERS - Change these as needed
# ==========================================
TICKER = "SPY"          # The ticker you want to analyze
LOOKBACK_YEARS = 5     # How many years of historical data to pull
INITIAL_CAPITAL = 100000
# ==========================================

# Fetch historical data
end_date = datetime.date.today()
start_date = end_date - datetime.timedelta(days=LOOKBACK_YEARS * 365)

print(f"Fetching data for {TICKER} from {start_date} to {end_date}...")
df = yf.download(TICKER, start=start_date, end=end_date, progress=False)

if df.empty:
    print("No data found. Please check the ticker symbol.")
else:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Calculate daily buy-and-hold returns
    df['Daily_Return'] = df['Close'].pct_change()
    df = df.dropna().copy()

    # Track position flags (1 = In Strategy Trade, 0 = Out/Cash)
    df['In_Trade'] = 0
    df['Year'] = df.index.year
    df['Month'] = df.index.month

    grouped = list(df.groupby(['Year', 'Month']))
    monthly_summary_records = []

    # Map out the continuous turn of the month holding windows
    for i in range(len(grouped) - 1):
        current_month_df = grouped[i][1]
        next_month_df = grouped[i+1][1]

        if len(current_month_df) >= 5 and len(next_month_df) >= 3:
            # Entry: Close of 5th-to-last day (captures returns for subsequent 4 days)
            last_5_days_indices = current_month_df.tail(5).index
            strategy_days_current = last_5_days_indices[1:]

            # Exit: Close of 3rd trading day of next month
            strategy_days_next = next_month_df.head(3).index

            # Combine into a unified window
            df.loc[strategy_days_current, 'In_Trade'] = 1
            df.loc[strategy_days_next, 'In_Trade'] = 1

            # Calculate combined compounded return for this specific month's window
            combined_window_returns = pd.concat([
                current_month_df.loc[strategy_days_current, 'Daily_Return'],
                next_month_df.loc[strategy_days_next, 'Daily_Return']
            ])
            tom_cum_return = (combined_window_returns + 1).prod() - 1

            monthly_summary_records.append({
                "Year": grouped[i][0][0],
                "Month": f"{grouped[i][0][1]:02d}",
                "ToM Window Return (%)": round(tom_cum_return * 100, 2)
            })

    # Calculate strategy returns and compounded equity curves
    df['Strategy_Return'] = df['Daily_Return'] * df['In_Trade']
    df['Buy_Hold_Equity'] = INITIAL_CAPITAL * (1 + df['Daily_Return']).cumprod()
    df['Strategy_Equity'] = INITIAL_CAPITAL * (1 + df['Strategy_Return']).cumprod()

    # Format and display the detailed monthly breakdown
    df_monthly_table = pd.DataFrame(monthly_summary_records).sort_values(['Year', 'Month'], ascending=False)

    print(f"\n{'='*15} CHRONOLOGICAL MONTHLY TOw RETURNS {'='*15}")
    # Display entire table by temporarily setting display options
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(df_monthly_table)

    # Calculate average return and win rate for ToM strategy (monthly level)
    avg_monthly_return = df_monthly_table['ToM Window Return (%)'].mean()
    monthly_win_rate = (df_monthly_table['ToM Window Return (%)'] > 0).sum() / len(df_monthly_table) * 100

    # Helper function for advanced risk metrics
    def calculate_metrics(equity_series, actual_return_series, in_trade_series=None):
        total_return = (equity_series.iloc[-1] / INITIAL_CAPITAL - 1) * 100

        # Sharpe Ratio calculation: use returns only when in trade for strategy, all returns for B&H
        if in_trade_series is not None:
            sharpe_calc_returns = actual_return_series[in_trade_series == 1]
        else:
            sharpe_calc_returns = actual_return_series

        if not sharpe_calc_returns.empty and sharpe_calc_returns.std() != 0:
            sharpe_ratio = (sharpe_calc_returns.mean() / sharpe_calc_returns.std()) * np.sqrt(252)
        else:
            sharpe_ratio = 0.0

        # Maximum Drawdown calculation
        peaks = equity_series.cummax()
        drawdowns = (equity_series - peaks) / peaks
        max_dd = drawdowns.min() * 100

        # Sortino Ratio calculation: use returns only when in trade for strategy, all returns for B&H
        sortino_calc_returns = sharpe_calc_returns # Same filtered returns as for Sharpe

        downside_returns = sortino_calc_returns[sortino_calc_returns < 0]
        if not downside_returns.empty and downside_returns.std() != 0:
            # Assuming risk-free rate is 0
            sortino_ratio = sortino_calc_returns.mean() / downside_returns.std() * np.sqrt(252)
        else:
            sortino_ratio = 0.0 # Handle cases with no downside returns or zero std

        return total_return, sharpe_ratio, max_dd, sortino_ratio

    # Calculate metrics for both profiles
    bh_ret, bh_sharpe, bh_dd, bh_sortino = calculate_metrics(df['Buy_Hold_Equity'], df['Daily_Return'])
    strat_ret, strat_sharpe, strat_dd, strat_sortino = calculate_metrics(df['Strategy_Equity'], df['Strategy_Return'], df['In_Trade'])

    time_exposure = (df['In_Trade'].sum() / len(df)) * 100

    # Risk-Adjusted Exposure: Total Return divided by Market Exposure (as a decimal)
    # For Buy & Hold, market exposure is 100%, so it's just total return.
    risk_adjusted_exposure_bh = bh_ret
    risk_adjusted_exposure_strat = strat_ret / (time_exposure / 100) if time_exposure > 0 else 0.0

    # Display Comprehensive Performance Summary Table
    print(f"\n{'='*15} COMPREHENSIVE PERFORMANCE SUMMARY {'='*15}")
    summary_data = {
        "Metric": [
            "Total Return",
            "Annualized Sharpe Ratio",
            "Annualized Sortino Ratio",
            "Max Drawdown",
            "Market Exposure",
            "Risk-Adjusted Exposure (Total Return / Market Exposure)",
            "Average Monthly ToM Return", # Only for ToM
            "Monthly ToM Win Rate"        # Only for ToM
        ],
        "Buy & Hold": [
            f"{bh_ret:.2f}%",
            f"{bh_sharpe:.2f}",
            f"{bh_sortino:.2f}",
            f"{bh_dd:.2f}%",
            "100.00%",
            f"{risk_adjusted_exposure_bh:.2f}%",
            "N/A", # Not applicable for B&H
            "N/A"  # Not applicable for B&H
        ],
        "Turn of the Month": [
            f"{strat_ret:.2f}%",
            f"{strat_sharpe:.2f}",
            f"{strat_sortino:.2f}",
            f"{strat_dd:.2f}%",
            f"{time_exposure:.2f}%",
            f"{risk_adjusted_exposure_strat:.2f}%",
            f"{avg_monthly_return:.2f}%",
            f"{monthly_win_rate:.2f}%"
        ]
    }
    combined_summary_df = pd.DataFrame(summary_data)
    display(combined_summary_df)

Fetching data for SPY from 2021-07-01 to 2026-06-30...

=============== CHRONOLOGICAL MONTHLY TOw RETURNS ===============


/tmp/ipykernel_1505/974400505.py:14: FutureWarning:

YF.download() has changed argument auto_adjust default to True



,Year,Month,ToM Window Return (%)
58,2026,05,1.15
57,2026,04,1.38
56,2026,03,0.32
55,2026,02,0.40
54,2026,01,-0.94
53,2025,12,0.21
52,2025,11,3.77
51,2025,10,-1.12
50,2025,09,1.23
49,2025,08,1.04



=============== COMPREHENSIVE PERFORMANCE SUMMARY ===============


,Metric,Buy & Hold,Turn of the Month
0,Total Return,84.19%,15.22%
1,Annualized Sharpe Ratio,0.80,0.61
2,Annualized Sortino Ratio,1.11,0.84
3,Max Drawdown,-24.50%,-15.49%
4,Market Exposure,100.00%,32.99%
5,Risk-Adjusted Exposure (Total Return / Market ...,84.19%,46.14%
6,Average Monthly ToM Return,N/A,0.27%
7,Monthly ToM Win Rate,N/A,55.93%


In [19]:
# Ensure data from cell 1 exists
if 'df' in locals() and not df.empty:
    # The metrics calculation and summary table display have been moved to the previous cell.
    # This cell now only handles plotting.

    # Plot the Portfolio Equity Chart
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index,
        y=df['Buy_Hold_Equity'],
        mode='lines',
        name='Buy & Hold Benchmark',
        line=dict(color='#ff7f0e', width=1.5)
    ))

    fig.add_trace(go.Scatter(
        x=df.index,
        y=df['Strategy_Equity'],
        mode='lines',
        name='Turn of the Month Strategy',
        line=dict(color='#2ca02c', width=2.5)
    ))

    fig.update_layout(
        title=f"Portfolio Equity Curve: {TICKER} Strategy Comparison",
        xaxis_title="Date",
        yaxis_title="Portfolio Value ($ USD)",
        template="plotly_dark",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        hovermode="x unified"
    )

    fig.show()
else:
    print("Please make sure to run Cell 1 successfully first to populate the trading data.")

In [20]:
print(f"\n{'='*15} ANALYZING 'FIRST DOWN DAY' (FDD) STRATEGY {'='*15}")

# Initialize 'In_Trade_FDD' column for the new strategy
df['In_Trade_FDD'] = 0
monthly_summary_records_FDD = []

for i in range(len(grouped) - 1):
    current_month_df = grouped[i][1]
    next_month_df = grouped[i+1][1]

    if len(next_month_df) >= 1: # Ensure there's a next month to exit
        # Define the 5-day window for entry search (last 5 trading days of current month N)
        entry_search_window_indices = current_month_df.tail(5).index

        entry_date = None
        # Find the first down day in this 5-day window
        for day_index in entry_search_window_indices:
            # A 'down day' means Daily_Return < 0
            if day_index in df.index and df.loc[day_index, 'Daily_Return'] < 0:
                entry_date = day_index
                break # Found the first down day

        if entry_date is not None:
            # Exit on the first trading day of the subsequent month (month N+1)
            exit_date = next_month_df.index[0]

            # Determine the actual trading days for this specific window
            # Strategy enters at the close of entry_date, so captures returns from the next day.
            # Strategy exits at the close of exit_date, so captures returns up to and including exit_date.
            trading_days = df.loc[entry_date:exit_date].index

            # Filter to include only days *after* entry_date (inclusive of next day's return)
            # and up to and including exit_date.
            valid_trading_days = trading_days[trading_days > entry_date]

            if not valid_trading_days.empty:
                df.loc[valid_trading_days, 'In_Trade_FDD'] = 1

                # Calculate compounded return for this specific FDD window
                fdd_cum_return = (df.loc[valid_trading_days, 'Daily_Return'] + 1).prod() - 1

                monthly_summary_records_FDD.append({
                    "Year": grouped[i][0][0],
                    "Month": f"{grouped[i][0][1]:02d}",
                    "FDD Window Return (%)": round(fdd_cum_return * 100, 2)
                })

# Calculate strategy returns and compounded equity curves for FDD
df['Strategy_Return_FDD'] = df['Daily_Return'] * df['In_Trade_FDD']
df['Strategy_Equity_FDD'] = INITIAL_CAPITAL * (1 + df['Strategy_Return_FDD']).cumprod()

# Format and display the detailed monthly breakdown for FDD
df_monthly_table_FDD = pd.DataFrame(monthly_summary_records_FDD).sort_values(['Year', 'Month'], ascending=False)

print(f"\n{'='*15} CHRONOLOGICAL MONTHLY FDD RETURNS {'='*15}")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_monthly_table_FDD)

# Calculate average return and win rate for FDD strategy (monthly level)
avg_monthly_return_FDD = df_monthly_table_FDD['FDD Window Return (%)'].mean()
monthly_win_rate_FDD = (df_monthly_table_FDD['FDD Window Return (%)'] > 0).sum() / len(df_monthly_table_FDD) * 100

# Calculate metrics for the new FDD strategy
fdd_ret, fdd_sharpe, fdd_dd, fdd_sortino = calculate_metrics(df['Strategy_Equity_FDD'], df['Strategy_Return_FDD'], df['In_Trade_FDD'])

time_exposure_FDD = (df['In_Trade_FDD'].sum() / len(df)) * 100
risk_adjusted_exposure_FDD = fdd_ret / (time_exposure_FDD / 100) if time_exposure_FDD > 0 else 0.0


# Update the comprehensive summary table to include FDD
print(f"\n{'='*15} UPDATED COMPREHENSIVE PERFORMANCE SUMMARY {'='*15}")
updated_summary_data = {
    "Metric": [
        "Total Return",
        "Annualized Sharpe Ratio",
        "Annualized Sortino Ratio",
        "Max Drawdown",
        "Market Exposure",
        "Risk-Adjusted Exposure (Total Return / Market Exposure)",
        "Average Monthly ToM Return",
        "Monthly ToM Win Rate"
    ],
    "Buy & Hold": [
        f"{bh_ret:.2f}%",
        f"{bh_sharpe:.2f}",
        f"{bh_sortino:.2f}",
        f"{bh_dd:.2f}%",
        "100.00%",
        f"{risk_adjusted_exposure_bh:.2f}%",
        "N/A",
        "N/A"
    ],
    "Turn of the Month": [
        f"{strat_ret:.2f}%",
        f"{strat_sharpe:.2f}",
        f"{strat_sortino:.2f}",
        f"{strat_dd:.2f}%",
        f"{time_exposure:.2f}%",
        f"{risk_adjusted_exposure_strat:.2f}%",
        f"{avg_monthly_return:.2f}%",
        f"{monthly_win_rate:.2f}%"
    ],
    "First Down Day": [
        f"{fdd_ret:.2f}%",
        f"{fdd_sharpe:.2f}",
        f"{fdd_sortino:.2f}",
        f"{fdd_dd:.2f}%",
        f"{time_exposure_FDD:.2f}%",
        f"{risk_adjusted_exposure_FDD:.2f}%",
        f"{avg_monthly_return_FDD:.2f}%",
        f"{monthly_win_rate_FDD:.2f}%"
    ]
}
updated_combined_summary_df = pd.DataFrame(updated_summary_data)
display(updated_combined_summary_df)


=============== ANALYZING 'FIRST DOWN DAY' (FDD) STRATEGY ===============

=============== CHRONOLOGICAL MONTHLY FDD RETURNS ===============


,Year,Month,FDD Window Return (%)
55,2026,05,1.08
54,2026,04,1.26
53,2026,03,1.57
52,2026,02,0.58
51,2026,01,-0.00
50,2025,12,-1.03
49,2025,10,0.52
48,2025,09,1.11
47,2025,08,-0.34
46,2025,07,-2.39



=============== UPDATED COMPREHENSIVE PERFORMANCE SUMMARY ===============


,Metric,Buy & Hold,Turn of the Month,First Down Day
0,Total Return,84.19%,15.22%,26.59%
1,Annualized Sharpe Ratio,0.80,0.61,1.72
2,Annualized Sortino Ratio,1.11,0.84,2.66
3,Max Drawdown,-24.50%,-15.49%,-10.25%
4,Market Exposure,100.00%,32.99%,17.73%
5,Risk-Adjusted Exposure (Total Return / Market ...,84.19%,46.14%,149.95%
6,Average Monthly ToM Return,N/A,0.27%,0.44%
7,Monthly ToM Win Rate,N/A,55.93%,60.71%


In [21]:
# Update the plotting cell to include the FDD strategy
# Ensure data from cell 1 exists
if 'df' in locals() and not df.empty:
    # Plot the Portfolio Equity Chart
    fig_updated = go.Figure()

    fig_updated.add_trace(go.Scatter(
        x=df.index,
        y=df['Buy_Hold_Equity'],
        mode='lines',
        name='Buy & Hold Benchmark',
        line=dict(color='#ff7f0e', width=1.5)
    ))

    fig_updated.add_trace(go.Scatter(
        x=df.index,
        y=df['Strategy_Equity'],
        mode='lines',
        name='Turn of the Month Strategy',
        line=dict(color='#2ca02c', width=2.5)
    ))

    fig_updated.add_trace(go.Scatter(
        x=df.index,
        y=df['Strategy_Equity_FDD'],
        mode='lines',
        name='First Down Day Strategy',
        line=dict(color='#1f77b4', width=2.5) # A distinct color for the new strategy
    ))

    fig_updated.update_layout(
        title=f"Portfolio Equity Curve: {TICKER} Strategy Comparison",
        xaxis_title="Date",
        yaxis_title="Portfolio Value ($ USD)",
        template="plotly_dark",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        hovermode="x unified"
    )

    fig_updated.show()
else:
    print("Please make sure to run Cell 1 successfully first to populate the trading data.")